<a href="https://colab.research.google.com/github/dsmirandax/data_augmentation/blob/main/C%C3%B3pia_de_imagenet_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Carregar imagenette

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install torchattacks

## Gerar ataque PGD no imagenette usando Resnet50

In [ ]:
"""
Gera ataques adversariais no Imagenette
"""

import os
import csv
import random
import torch
import torch.nn as nn
from torchvision import transforms, models
from torchvision.datasets import Imagenette
from torch.utils.data import DataLoader, Subset
from PIL import Image

import torchattacks

# ---------------------------------------------------------------- config
# No Colab, monte o Drive antes de rodar:
#     from google.colab import drive
#     drive.mount('/content/drive')
PASTA_DRIVE = "/content/drive/MyDrive/adversarial_heatmap"
PASTA_TRABALHO = PASTA_DRIVE if os.path.isdir("/content/drive/MyDrive") else "."
os.makedirs(PASTA_TRABALHO, exist_ok=True)

RAIZ_SAIDA = os.path.join(PASTA_TRABALHO, "imagenette_atacado")
RESOLUCAO = 224          # entrada da ResNet50
N_TESTE = 400            # imagens no split test (clean + cada ataque)
N_TREINO = 400           # imagens no split train (só clean, p/ calibração)
BATCH_SIZE = 16
EPS = 4 / 255            # perturbação Linf para bim/pgd (iterativos)
EPS_FGSM = 8 / 255       # FGSM é de passo único -- precisa de mais orçamento
                         # para causar dano comparável (com 4/255 a acurácia
                         # ficou em 0,94, praticamente sem dano)

# Imagenette = 10 classes do ImageNet. Índices na saída de 1000 classes:
IMAGENETTE_PARA_IMAGENET = {
    0: 0,     # tench
    1: 217,   # English springer
    2: 482,   # cassette player
    3: 491,   # chain saw
    4: 497,   # church
    5: 566,   # French horn
    6: 569,   # garbage truck
    7: 571,   # gas pump
    8: 574,   # golf ball
    9: 701,   # parachute
}

# ---------------------------------------------------- modelo (f_theta)


class ResNet50Imagenette(nn.Module):
    """ResNet50 do ImageNet, restrita às 10 classes do Imagenette.

    Recebe entrada em [0,1] e normaliza por dentro -- necessário porque
    os ataques operam no espaço de pixel [0,1], não no normalizado.
    """

    def __init__(self):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.eval()
        indices = [IMAGENETTE_PARA_IMAGENET[i] for i in range(10)]
        self.register_buffer("indices_imagenette", torch.tensor(indices))
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("desvio", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x_01):
        x_norm = (x_01 - self.media) / self.desvio
        logits_1000 = self.backbone(x_norm)
        return logits_1000[:, self.indices_imagenette]  # (B, 10)


def _rotulos_do_dataset(ds):
    """Tenta ler os rótulos sem carregar as imagens (rápido)."""
    for atributo in ("_samples", "samples", "imgs"):
        if hasattr(ds, atributo):
            try:
                return [int(s[1]) for s in getattr(ds, atributo)]
            except (TypeError, IndexError, ValueError):
                continue
    return None


def carregar_imagenette(split, n_imagens, seed=42):
    """Se sua torchvision for < 0.18 e não tiver datasets.Imagenette,
    baixe o tarball manualmente e troque por ImageFolder:
        https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
        ds = torchvision.datasets.ImageFolder(caminho, transform=tf)

    IMPORTANTE: o Imagenette vem ORDENADO POR CLASSE. Pegar os primeiros
    N índices daria só a classe 0 -- por isso a amostragem abaixo é
    estratificada (n/10 por classe), com sorteio aleatório dentro de cada.
    """
    tf = transforms.Compose([
        transforms.Resize((RESOLUCAO, RESOLUCAO)),
        transforms.ToTensor(),  # já deixa em [0,1]
    ])
    ds = Imagenette(root="./dados_imagenette", split=split, size="160px",
                    download=True, transform=tf)

    rng = random.Random(seed)
    rotulos = _rotulos_do_dataset(ds)

    if rotulos is None:
        # fallback: permutação aleatória -- aproximadamente balanceada
        print("   (rótulos não acessíveis; usando permutação aleatória)", flush=True)
        indices = list(range(len(ds)))
        rng.shuffle(indices)
        return Subset(ds, indices[:min(n_imagens, len(ds))])

    # amostragem estratificada: n_imagens/10 de cada classe
    por_classe = {}
    for idx, rotulo in enumerate(rotulos):
        por_classe.setdefault(rotulo, []).append(idx)

    n_por_classe = max(1, n_imagens // len(por_classe))
    selecionados = []
    for rotulo in sorted(por_classe):
        disponiveis = por_classe[rotulo]
        rng.shuffle(disponiveis)
        selecionados.extend(disponiveis[:n_por_classe])

    rng.shuffle(selecionados)
    print(f"   ({split}: {len(selecionados)} imagens, "
          f"~{n_por_classe} por classe, {len(por_classe)} classes)", flush=True)
    return Subset(ds, selecionados)


# ------------------------------------------------------------- salvar


def salvar_lote(x_01, rotulos, pasta, prefixo, indice_inicial):
    os.makedirs(pasta, exist_ok=True)
    to_pil = transforms.ToPILImage()
    linhas = []
    for i in range(x_01.shape[0]):
        nome = f"{prefixo}_{indice_inicial + i:05d}.png"
        to_pil(x_01[i].cpu().clamp(0, 1)).save(os.path.join(pasta, nome))
        linhas.append((nome, int(rotulos[i])))
    return linhas


def escrever_csv(caminho, registros):
    """registros: lista de (nome_arquivo, attack_type, original_label)"""
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    with open(caminho, "w", newline="") as f:
        escritor = csv.writer(f)
        escritor.writerow(["image_path", "attack_type", "original_label"])
        for nome, tipo, rotulo in registros:
            escritor.writerow([f"{tipo}/{nome}", tipo, rotulo])
    print(f"   csv escrito: {caminho}  ({len(registros)} linhas)", flush=True)


# --------------------------------------------------------------- main


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f">> device = {device}", flush=True)

    modelo = ResNet50Imagenette().to(device).eval()
    for p in modelo.parameters():
        p.requires_grad_(False)

    # ---- checagem de sanidade: classes presentes + mapeamento correto ----
    ds_teste = carregar_imagenette("val", N_TESTE)

    corretos, total = 0, 0
    rotulos_vistos = set()
    with torch.no_grad():
        for x, y in DataLoader(ds_teste, batch_size=BATCH_SIZE):
            rotulos_vistos.update(y.tolist())
            pred = modelo(x.to(device)).argmax(dim=1).cpu()
            corretos += (pred == y).sum().item()
            total += y.size(0)

    print(f">> classes presentes na amostra: {len(rotulos_vistos)} de 10", flush=True)
    if len(rotulos_vistos) < 10:
        print("   !! AMOSTRA DESBALANCEADA -- esperado as 10 classes.", flush=True)
        print(f"   !! presentes: {sorted(rotulos_vistos)}", flush=True)
        return

    acc_clean = corretos / total
    print(f">> acc clean (val) = {acc_clean:.4f}  ({total} imagens)", flush=True)
    if acc_clean < 0.7:
        print("   !! ACC BAIXA -- provável erro no mapeamento IMAGENETTE_PARA_IMAGENET.", flush=True)
        print("   !! Confira os índices antes de gerar os ataques.", flush=True)
        return
    print("   (acc alta confirma que o mapeamento de classes está correto)", flush=True)

    ataques = {
        "fgsm": torchattacks.FGSM(modelo, eps=EPS_FGSM),
        "bim": torchattacks.BIM(modelo, eps=EPS, alpha=EPS / 4, steps=10),
        "pgd": torchattacks.PGD(modelo, eps=EPS, alpha=EPS / 4, steps=20),
        "cw": torchattacks.CW(modelo, c=1, kappa=10, steps=50, lr=0.01),
        "deepfool": torchattacks.DeepFool(modelo, steps=50),
    }

    # =============== SPLIT TEST: clean + todos os ataques ===============
    raiz_test = os.path.join(RAIZ_SAIDA, "test")
    registros_por_tipo = {t: [] for t in ["clean", *ataques.keys()]}
    acuracias = {}

    print("\n>> gerando split TEST...", flush=True)
    for tipo in ["clean", *ataques.keys()]:
        print(f"   -- {tipo} --", flush=True)
        corretos, total, idx = 0, 0, 0
        for x, y in DataLoader(ds_teste, batch_size=BATCH_SIZE):
            x, y = x.to(device), y.to(device)
            x_saida = x if tipo == "clean" else ataques[tipo](x, y)

            with torch.no_grad():
                pred = modelo(x_saida).argmax(dim=1)
            corretos += (pred == y).sum().item()
            total += y.size(0)

            linhas = salvar_lote(x_saida, y, os.path.join(raiz_test, tipo), tipo, idx)
            registros_por_tipo[tipo].extend([(nome, tipo, rot) for nome, rot in linhas])
            idx += x.shape[0]

        acuracias[tipo] = corretos / total
        print(f"      acc sob {tipo} = {acuracias[tipo]:.4f}", flush=True)

    # CSVs com os MESMOS nomes que nossos scripts já esperam
    escrever_csv(os.path.join(raiz_test, "metadata_test(fgsm, bim, cANDw).csv"),
                 registros_por_tipo["clean"] + registros_por_tipo["fgsm"] +
                 registros_por_tipo["bim"] + registros_por_tipo["cw"])
    escrever_csv(os.path.join(raiz_test, "metadata_test(pgd, backdoor).csv"),
                 registros_por_tipo["pgd"])
    escrever_csv(os.path.join(raiz_test, "metadata_test(deepfool).csv"),
                 registros_por_tipo["deepfool"])

    # ============ SPLIT TRAIN: só clean (para calibração) ============
    print("\n>> gerando split TRAIN (só clean, para calibração)...", flush=True)
    ds_treino = carregar_imagenette("train", N_TREINO)
    raiz_train = os.path.join(RAIZ_SAIDA, "train")
    registros_treino, idx = [], 0
    for x, y in DataLoader(ds_treino, batch_size=BATCH_SIZE):
        linhas = salvar_lote(x, y, os.path.join(raiz_train, "clean"), "clean", idx)
        registros_treino.extend([(nome, "clean", rot) for nome, rot in linhas])
        idx += x.shape[0]
    escrever_csv(os.path.join(raiz_train, "metadata_train(fgsm, bim, c and w).csv"),
                 registros_treino)

    # ------------------------------------------------------- resumo
    eps_por_ataque = {"clean": "-", "fgsm": f"{EPS_FGSM * 255:.0f}/255",
                       "bim": f"{EPS * 255:.0f}/255", "pgd": f"{EPS * 255:.0f}/255",
                       "cw": "(sem eps)", "deepfool": "(sem eps)"}
    print(f"\n{'tipo':10s}  {'eps':>10s}  {'acurácia':>10s}  {'dano (1-acc)':>13s}", flush=True)
    for tipo, acc in acuracias.items():
        print(f"{tipo:10s}  {eps_por_ataque.get(tipo, '?'):>10s}  {acc:10.4f}  {1 - acc:13.4f}", flush=True)

    print(f"\n>> pronto. Aponte caminho_dataset.py para: {os.path.abspath(RAIZ_SAIDA)}", flush=True)
    print(">> ATENÇÃO: imagens agora são 224x224, não 32x32.", flush=True)
    print("   Nos scripts de detecção, ajuste TAMANHO_PATCH de 4 para 28", flush=True)
    print("   (224/28 = grade 8x8, mesma granularidade relativa de antes).", flush=True)


if __name__ == "__main__":
    main()

>> device = cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 75.9MB/s]
100%|██████████| 99.0M/99.0M [00:02<00:00, 39.8MB/s]


   (val: 400 imagens, ~40 por classe, 10 classes)
>> classes presentes na amostra: 10 de 10
>> acc clean (val) = 0.9925  (400 imagens)
   (acc alta confirma que o mapeamento de classes está correto)

>> gerando split TEST...
   -- clean --
      acc sob clean = 0.9925
   -- fgsm --
      acc sob fgsm = 0.8400
   -- bim --
      acc sob bim = 0.0475
   -- pgd --
      acc sob pgd = 0.0125
   -- cw --
      acc sob cw = 0.0000
   -- deepfool --
      acc sob deepfool = 0.2425
   csv escrito: /content/drive/MyDrive/adversarial_heatmap/imagenette_atacado/test/metadata_test(fgsm, bim, cANDw).csv  (1600 linhas)
   csv escrito: /content/drive/MyDrive/adversarial_heatmap/imagenette_atacado/test/metadata_test(pgd, backdoor).csv  (400 linhas)
   csv escrito: /content/drive/MyDrive/adversarial_heatmap/imagenette_atacado/test/metadata_test(deepfool).csv  (400 linhas)

>> gerando split TRAIN (só clean, para calibração)...
   (train: 400 imagens, ~40 por classe, 10 classes)
   csv escrito: /content/

## Teste detecção orientada por heatmap e oclusão

In [2]:
# COMUM
"""
Módulo comum: dataset, f_theta e a cabeça de localização.
"""

import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Dataset
from PIL import Image

# ------------------------------------------------------- persistência
# No Colab, monte o Drive antes de importar este módulo:
#     from google.colab import drive
#     drive.mount('/content/drive')
# Tudo (dataset gerado, cabeça treinada, resultados) passa a ser salvo
# em PASTA_TRABALHO, que sobrevive a desconexões do notebook.

PASTA_DRIVE = "/content/drive/MyDrive/adversarial_heatmap"


def _resolver_pasta_trabalho():
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(PASTA_DRIVE, exist_ok=True)
        return PASTA_DRIVE
    return "."


PASTA_TRABALHO = _resolver_pasta_trabalho()

RAIZ_DADOS = os.path.join(PASTA_TRABALHO, "imagenette_atacado")
CAMINHO_CABECA = os.path.join(PASTA_TRABALHO, "cabeca_localizacao.pt")
RESOLUCAO = 224
N_CLASSES = 10

CLASSES_IMAGENETTE = ["tench", "English springer", "cassette player", "chain saw",
                       "church", "French horn", "garbage truck", "gas pump",
                       "golf ball", "parachute"]

IMAGENETTE_PARA_IMAGENET = {0: 0, 1: 217, 2: 482, 3: 491, 4: 497,
                             5: 566, 6: 569, 7: 571, 8: 574, 9: 701}

CSV_TEST_PRINCIPAL = "metadata_test(fgsm, bim, cANDw).csv"
CSV_TEST_PGD = "metadata_test(pgd, backdoor).csv"
CSV_TRAIN = "metadata_train(fgsm, bim, c and w).csv"


# ----------------------------------------------------------------- dados

class DatasetAtacado(Dataset):
    """Lê a estrutura gerada por gerar_ataques_imagenette.py."""

    def __init__(self, split, nome_csv, attack_type, n_max=None):
        raiz_split = os.path.join(RAIZ_DADOS, split)
        df = pd.read_csv(os.path.join(raiz_split, nome_csv))
        df = df[df["attack_type"] == attack_type].reset_index(drop=True)
        if n_max is not None:
            df = df.iloc[:n_max].reset_index(drop=True)
        self.df = df
        self.raiz_split = raiz_split
        self.tf = transforms.Compose([
            transforms.Resize((RESOLUCAO, RESOLUCAO)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        linha = self.df.iloc[idx]
        caminho = os.path.join(self.raiz_split, linha["image_path"])
        img = Image.open(caminho).convert("RGB")
        return self.tf(img), int(linha["original_label"])


# --------------------------------------------------------------- f_theta

class ResNet50Imagenette(nn.Module):
    """Mesmo f_theta usado para gerar os ataques -- entrada em [0,1]."""

    def __init__(self):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.eval()
        indices = [IMAGENETTE_PARA_IMAGENET[i] for i in range(N_CLASSES)]
        self.register_buffer("indices", torch.tensor(indices))
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("desvio", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        # extrator até a última camada convolucional (features espaciais 7x7)
        self.extrator = nn.Sequential(*list(self.backbone.children())[:-2])

    def normalizar(self, x_01):
        return (x_01 - self.media) / self.desvio

    def forward(self, x_01):
        return self.backbone(self.normalizar(x_01))[:, self.indices]

    def features_espaciais(self, x_01):
        """(B, 2048, 7, 7) -- entrada para a cabeça de localização."""
        return self.extrator(self.normalizar(x_01))


# ------------------------------------------------- cabeça de localização

class CabecaLocalizacao(nn.Module):
    """Produz um mapa espacial condicionado à classe.

    Treinada por GARGALO: a classificação só enxerga o que o mapa deixa
    passar, então o mapa é obrigado a apontar onde está a evidência real
    da classe -- sem nunca receber anotação de região.
    """

    def __init__(self, dim_features=2048, n_classes=N_CLASSES, dim_emb=64):
        super().__init__()
        self.emb_classe = nn.Embedding(n_classes, dim_emb)
        self.conv = nn.Sequential(
            nn.Conv2d(dim_features + dim_emb, 256, kernel_size=1), nn.ReLU(),
            nn.Conv2d(256, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1),
        )
        self.classificador = nn.Linear(dim_features, n_classes)

    def forward(self, features, classe):
        B, C, H, W = features.shape
        emb = self.emb_classe(classe).unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)
        mapa = torch.sigmoid(self.conv(torch.cat([features, emb], dim=1)))  # (B,1,H,W)

        # média PONDERADA, não simples: normaliza pela massa do mapa.
        # Sem isso, encolher o mapa encolhe o vetor de features junto, e a
        # rede colapsa o mapa para zero como atalho para minimizar a
        # penalidade de esparsidade (o sigmoide satura e o treino morre).
        soma = (features * mapa).sum(dim=[2, 3])
        massa = mapa.sum(dim=[2, 3]).clamp(min=1e-6)
        vetor = soma / massa
        return self.classificador(vetor), mapa


def carregar_cabeca(device):
    cabeca = CabecaLocalizacao().to(device)
    cabeca.load_state_dict(torch.load(CAMINHO_CABECA, map_location=device))
    cabeca.eval()
    return cabeca


def mapa_para_resolucao(mapa, resolucao=RESOLUCAO):
    """Interpola o mapa 7x7 para o tamanho da imagem."""
    return F.interpolate(mapa, size=(resolucao, resolucao), mode="bilinear", align_corners=False)

In [ ]:
# ETAPA 1
"""
ETAPA 1 -- verifica o dataset gerado e mede a acurácia de clean e PGD.

"""

import torch
from torch.utils.data import DataLoader

from comum import (DatasetAtacado, ResNet50Imagenette, CLASSES_IMAGENETTE,
                    CSV_TEST_PRINCIPAL, CSV_TEST_PGD, CSV_TRAIN)


def acuracia(modelo, ds, device, batch_size=16):
    corretos, total = 0, 0
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=batch_size):
            pred = modelo(x.to(device)).argmax(dim=1).cpu()
            corretos += (pred == y).sum().item()
            total += y.size(0)
    return corretos / total, total


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f">> device = {device}", flush=True)

    modelo = ResNet50Imagenette().to(device).eval()
    for p in modelo.parameters():
        p.requires_grad_(False)

    conjuntos = [
        ("test/clean", "test", CSV_TEST_PRINCIPAL, "clean"),
        ("test/pgd", "test", CSV_TEST_PGD, "pgd"),
        ("train/clean", "train", CSV_TRAIN, "clean"),
    ]

    print(f"\n{'conjunto':14s}  {'n':>5s}  {'acurácia':>9s}", flush=True)
    for nome, split, csv, tipo in conjuntos:
        try:
            ds = DatasetAtacado(split, csv, tipo)
            acc, n = acuracia(modelo, ds, device)
            print(f"{nome:14s}  {n:5d}  {acc:9.4f}", flush=True)
        except FileNotFoundError as e:
            print(f"{nome:14s}  ERRO: {e}", flush=True)
            return

    # distribuição de classes -- a cabeça precisa ver as 10
    ds_treino = DatasetAtacado("train", CSV_TRAIN, "clean")
    contagem = ds_treino.df["original_label"].value_counts().sort_index()
    print(f"\n>> distribuição de classes em train/clean:", flush=True)
    for classe_id, qtd in contagem.items():
        print(f"   {classe_id} {CLASSES_IMAGENETTE[classe_id]:20s}  {qtd:4d}", flush=True)

    if len(contagem) < 10:
        print(f"\n   !! só {len(contagem)} classes presentes -- a cabeça precisa das 10.", flush=True)
        print("   !! aumente N_TREINO em gerar_ataques_imagenette.py e gere de novo.", flush=True)
    else:
        print(f"\n>> ok, as 10 classes estão presentes. Pode rodar a etapa 2.", flush=True)


if __name__ == "__main__":
    main()

>> device = cuda

conjunto            n   acurácia
test/clean        400     0.9925
test/pgd          400     0.0125
train/clean       400     0.9975

>> distribuição de classes em train/clean:
   0 tench                   40
   1 English springer        40
   2 cassette player         40
   3 chain saw               40
   4 church                  40
   5 French horn             40
   6 garbage truck           40
   7 gas pump                40
   8 golf ball               40
   9 parachute               40

>> ok, as 10 classes estão presentes. Pode rodar a etapa 2.


In [ ]:
#ETAPA 2
"""
ETAPA 2 -- treina a cabeça de localização, SÓ em imagens limpas.

Três termos na perda:
  1. classificação: com a classe VERDADEIRA, o mapa tem que deixar passar
     evidência suficiente para acertar -- é o gargalo que força o mapa a
     apontar onde está o objeto.
  2. esparsidade: penaliza mapa grande, para ele não "abrir tudo".
  3. margem: com uma classe ERRADA, o mapa deve ser MENOR que o do certo
     por uma margem. É esse termo que faz o mapa virar sinal de detecção:
     numa imagem adversarial, a evidência de y_hat não existe de verdade,
     então o mapa deve sair fraco.

"""

import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from comum import (DatasetAtacado, ResNet50Imagenette, CabecaLocalizacao,
                    CAMINHO_CABECA, CSV_TRAIN, N_CLASSES, PASTA_TRABALHO)

N_EPOCAS = 20
BATCH_SIZE = 16
LR = 1e-3
PESO_ESPARSIDADE = 0.1
PESO_NEGATIVO = 1.0
MARGEM = 0.15
EPOCAS_AQUECIMENTO = 2
FRACAO_VALIDACAO = 0.2
SEED = 42

CAMINHO_CHECKPOINT = os.path.join(PASTA_TRABALHO, "cabeca_checkpoint.pt")


def concentracao_do_mapa(mapa, fracao_top=0.25):
    """Fração da massa total que está nos top-k% pixels mais quentes.
    Mapa uniforme -> fracao_top. Mapa bem localizado -> perto de 1.0."""
    B = mapa.shape[0]
    achatado = mapa.view(B, -1)
    k = max(1, int(achatado.shape[1] * fracao_top))
    topk = torch.topk(achatado, k, dim=1).values
    return (topk.sum(dim=1) / achatado.sum(dim=1).clamp(min=1e-6)).mean()


@torch.no_grad()
def avaliar(cabeca, f_theta, loader, device):
    """Métricas em dado limpo nunca visto no treino."""
    somas = {"acc": 0.0, "certo": 0.0, "errado": 0.0, "conc": 0.0, "contraste": 0.0}
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        feats = f_theta.features_espaciais(x)

        logits, mapa_certo = cabeca(feats, y)
        deslocamento = torch.randint(1, N_CLASSES, y.shape, device=device)
        _, mapa_errado = cabeca(feats, (y + deslocamento) % N_CLASSES)

        somas["acc"] += (logits.argmax(dim=1) == y).float().mean().item()
        somas["certo"] += mapa_certo.mean().item()
        somas["errado"] += mapa_errado.mean().item()
        somas["conc"] += concentracao_do_mapa(mapa_certo).item()
        somas["contraste"] += mapa_certo.view(mapa_certo.shape[0], -1).std(dim=1).mean().item()
        n += 1
    return {k: v / n for k, v in somas.items()}


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f">> device = {device}", flush=True)
    print(f">> pasta de trabalho: {PASTA_TRABALHO}", flush=True)

    f_theta = ResNet50Imagenette().to(device).eval()
    for p in f_theta.parameters():
        p.requires_grad_(False)

    cabeca = CabecaLocalizacao().to(device)
    otimizador = torch.optim.Adam(cabeca.parameters(), lr=LR)

    epoca_inicial = 0
    if os.path.exists(CAMINHO_CHECKPOINT):
        ckpt = torch.load(CAMINHO_CHECKPOINT, map_location=device)
        cabeca.load_state_dict(ckpt["cabeca"])
        otimizador.load_state_dict(ckpt["otimizador"])
        epoca_inicial = ckpt["epoca"]
        print(f">> checkpoint encontrado -- retomando da época {epoca_inicial + 1}", flush=True)

    if epoca_inicial >= N_EPOCAS:
        print(">> treino já concluído. Apague o checkpoint para treinar do zero.", flush=True)
        return

    # split treino/validação -- ambos LIMPOS, validação nunca treinada
    ds_completo = DatasetAtacado("train", CSV_TRAIN, "clean")
    n_val = int(len(ds_completo) * FRACAO_VALIDACAO)
    n_treino = len(ds_completo) - n_val
    ds_treino, ds_val = random_split(ds_completo, [n_treino, n_val],
                                      generator=torch.Generator().manual_seed(SEED))
    loader = DataLoader(ds_treino, batch_size=BATCH_SIZE, shuffle=True)
    loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE)
    print(f">> treino: {n_treino} imagens  |  validação: {n_val} imagens (limpas, não treinadas)", flush=True)

    historico = []
    for epoca in range(epoca_inicial, N_EPOCAS):
        peso_esp = 0.0 if epoca < EPOCAS_AQUECIMENTO else PESO_ESPARSIDADE
        cabeca.train()
        acc_treino, n_lotes = 0.0, 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                feats = f_theta.features_espaciais(x)

            logits, mapa_certo = cabeca(feats, y)
            perda_clf = F.cross_entropy(logits, y)
            media_certo = mapa_certo.mean()

            deslocamento = torch.randint(1, N_CLASSES, y.shape, device=device)
            _, mapa_errado = cabeca(feats, (y + deslocamento) % N_CLASSES)
            perda_neg = F.relu(mapa_errado.mean() - media_certo + MARGEM)

            perda = perda_clf + peso_esp * media_certo + PESO_NEGATIVO * perda_neg
            otimizador.zero_grad()
            perda.backward()
            otimizador.step()

            acc_treino += (logits.argmax(dim=1) == y).float().mean().item()
            n_lotes += 1

        cabeca.eval()
        v = avaliar(cabeca, f_theta, loader_val, device)
        razao = v["certo"] / max(v["errado"], 1e-6)
        historico.append({"epoca": epoca + 1, **v, "razao": razao,
                           "acc_treino": acc_treino / n_lotes})

        marca = "  (aquecimento)" if epoca < EPOCAS_AQUECIMENTO else ""
        print(f"   ép {epoca + 1:2d}/{N_EPOCAS}: acc_tr={acc_treino / n_lotes:.4f}  "
              f"acc_val={v['acc']:.4f}  certo={v['certo']:.4f}  errado={v['errado']:.4f}  "
              f"razão={razao:.2f}x  concentração={v['conc']:.3f}  "
              f"contraste={v['contraste']:.4f}{marca}", flush=True)

        if v["certo"] < 0.01:
            print("   !! MAPA COLAPSANDO -- interrompendo.", flush=True)
            return

        torch.save({"cabeca": cabeca.state_dict(),
                    "otimizador": otimizador.state_dict(),
                    "epoca": epoca + 1}, CAMINHO_CHECKPOINT)

    torch.save(cabeca.state_dict(), CAMINHO_CABECA)
    print(f">> cabeça salva em {CAMINHO_CABECA}", flush=True)

    # ------------------------------------------------------- diagnóstico
    print("\n>> DIAGNÓSTICO (tudo em dado limpo -- nada de adversarial aqui):", flush=True)

    ultima = historico[-1]
    lacuna = ultima["acc_treino"] - ultima["acc"]
    print(f"   acc treino={ultima['acc_treino']:.4f}  acc validação={ultima['acc']:.4f}  "
          f"lacuna={lacuna:+.4f}", flush=True)
    if lacuna > 0.05:
        print("      -> lacuna grande: a cabeça está DECORANDO. Mais imagens de treino.", flush=True)
    else:
        print("      -> lacuna pequena: generaliza (não é memorização).", flush=True)

    print(f"   concentração final={ultima['conc']:.3f}  (0.25 = mapa uniforme, 1.0 = pontual)", flush=True)
    if ultima["conc"] < 0.35:
        print("      -> mapa QUASE UNIFORME: não está localizando nada.", flush=True)
        print("         A oclusão 'guiada' escolheria blocos ~aleatórios.", flush=True)
        print("         Suba PESO_ESPARSIDADE (ex. 0.3, 0.5) e treine de novo.", flush=True)
    elif ultima["conc"] < 0.6:
        print("      -> localização moderada.", flush=True)
    else:
        print("      -> mapa bem localizado.", flush=True)

    melhor = max(historico, key=lambda h: h["razao"])
    print(f"   melhor razão certo/errado: {melhor['razao']:.2f}x na época {melhor['epoca']}", flush=True)
    if melhor["epoca"] == N_EPOCAS:
        print("      -> a razão ainda subia no fim: aumente N_EPOCAS.", flush=True)

    print("\n   Escolha os hiperparâmetros SÓ por estes números, congele-os,", flush=True)
    print("   e só então rode a etapa 3. Não volte aqui depois de ver o AUROC.", flush=True)


if __name__ == "__main__":
    main()

>> device = cuda
>> pasta de trabalho: /content/drive/MyDrive/adversarial_heatmap
>> treino: 1600 imagens  |  validação: 400 imagens (limpas, não treinadas)
   ép  1/20: acc_tr=0.8994  acc_val=0.9875  certo=0.9629  errado=0.7387  razão=1.30x  concentração=0.256  contraste=0.0898  (aquecimento)
   ép  2/20: acc_tr=0.9931  acc_val=0.9975  certo=0.9729  errado=0.7279  razão=1.34x  concentração=0.253  contraste=0.0693  (aquecimento)
   ép  3/20: acc_tr=0.9969  acc_val=0.9975  certo=0.9156  errado=0.6482  razão=1.41x  concentração=0.274  contraste=0.1520
   ép  4/20: acc_tr=0.9981  acc_val=1.0000  certo=0.8936  errado=0.6358  razão=1.41x  concentração=0.281  contraste=0.1817
   ép  5/20: acc_tr=0.9981  acc_val=1.0000  certo=0.8753  errado=0.6093  razão=1.44x  concentração=0.290  contraste=0.2018
   ép  6/20: acc_tr=0.9994  acc_val=1.0000  certo=0.9284  errado=0.6599  razão=1.41x  concentração=0.269  contraste=0.1433
   ép  7/20: acc_tr=0.9994  acc_val=1.0000  certo=0.8321  errado=0.5560  ra

In [ ]:
#ETAPA 3
"""
ETAPA 3 -- gera N amostras aumentadas por imagem, guiadas pelo heatmap,
com DUAS transformações lado a lado (oclusão e recorte), e reporta:
  - voto majoritário (robustez: o voto corrige o rótulo?)
  - taxa de concordância com y_hat (detecção: clean concorda mais?)
  - força/dispersão do mapa (sinal que não precisa de augmentation nenhuma)
  - histograma de classes das N amostras aumentadas
"""

import os
import numpy as np
import pandas as pd
import torch

from comum import (DatasetAtacado, ResNet50Imagenette, carregar_cabeca,
                    mapa_para_resolucao, CLASSES_IMAGENETTE,
                    CSV_TEST_PRINCIPAL, CSV_TEST_PGD, RESOLUCAO, PASTA_TRABALHO)

# ----------------------------------------------------- PARÂMETROS
# Edite aqui (em notebook não se passa argumento de linha de comando).
N_AMOSTRAS = 100        # amostras aumentadas por imagem
N_IMAGENS = 40          # imagens por grupo (clean e pgd)
SEED = 42

TAMANHO_BLOCO = 16          # 224/16 = grade 14x14 de blocos
FRACAO_BLOCOS_MEXIDOS = 0.5  # dos blocos salientes, quantos são afetados por amostra
LOTE_VARIANTES = 50          # quantas variantes classificar por vez


def blocos_salientes(mapa_224, tamanho_bloco, fracao_top=0.25):
    """Divide em blocos e devolve os índices dos blocos mais quentes."""
    n_lado = RESOLUCAO // tamanho_bloco
    mapa_blocos = mapa_224.view(n_lado, tamanho_bloco, n_lado, tamanho_bloco).mean(dim=[1, 3])
    n_top = max(1, int(n_lado * n_lado * fracao_top))
    indices = torch.topk(mapa_blocos.flatten(), n_top).indices
    return indices, n_lado


def gerar_variantes(x, indices_salientes, n_lado, n_amostras, modo, tamanho_bloco, gerador):
    """Gera n_amostras variantes. Cada uma sorteia um subconjunto diferente
    dos blocos salientes -- é daí que vem a diversidade das N amostras."""
    variantes = []
    n_mexidos = max(1, int(len(indices_salientes) * FRACAO_BLOCOS_MEXIDOS))

    for _ in range(n_amostras):
        perm = torch.randperm(len(indices_salientes), generator=gerador)[:n_mexidos]
        escolhidos = indices_salientes[perm]

        mascara = torch.ones(n_lado * n_lado, device=x.device) if modo == "oclusao" \
            else torch.zeros(n_lado * n_lado, device=x.device)
        mascara[escolhidos] = 0.0 if modo == "oclusao" else 1.0

        mascara = mascara.view(1, 1, n_lado, n_lado)
        mascara = mascara.repeat_interleave(tamanho_bloco, dim=2).repeat_interleave(tamanho_bloco, dim=3)
        variantes.append(x * mascara)

    return torch.cat(variantes, dim=0)


def processar_grupo(nome_grupo, ds, f_theta, cabeca, device, n_amostras, gerador):
    linhas = []
    for i in range(len(ds)):
        x, rotulo_verdadeiro = ds[i]
        x = x.unsqueeze(0).to(device)

        with torch.no_grad():
            y_hat = f_theta(x).argmax(dim=1).item()
            feats = f_theta.features_espaciais(x)
            _, mapa = cabeca(feats, torch.tensor([y_hat], device=device))
            mapa_224 = mapa_para_resolucao(mapa)[0, 0]

        forca_mapa = mapa_224.mean().item()
        dispersao_mapa = mapa_224.std().item()
        indices, n_lado = blocos_salientes(mapa_224, TAMANHO_BLOCO)

        registro = {
            "grupo": nome_grupo, "idx": i,
            "rotulo_verdadeiro": rotulo_verdadeiro, "y_hat": y_hat,
            "y_hat_correto": int(y_hat == rotulo_verdadeiro),
            "forca_mapa": forca_mapa, "dispersao_mapa": dispersao_mapa,
        }

        for modo in ["oclusao", "recorte"]:
            variantes = gerar_variantes(x, indices, n_lado, n_amostras, modo, TAMANHO_BLOCO, gerador)
            preds = []
            with torch.no_grad():
                for j in range(0, len(variantes), LOTE_VARIANTES):
                    preds.append(f_theta(variantes[j:j + LOTE_VARIANTES]).argmax(dim=1).cpu())
            preds = torch.cat(preds)

            contagem = torch.bincount(preds, minlength=len(CLASSES_IMAGENETTE))
            voto = contagem.argmax().item()
            registro[f"{modo}_concordancia"] = (preds == y_hat).float().mean().item()
            registro[f"{modo}_voto"] = voto
            registro[f"{modo}_voto_correto"] = int(voto == rotulo_verdadeiro)
            registro[f"{modo}_n_classes_distintas"] = int((contagem > 0).sum().item())
            registro[f"{modo}_histograma"] = "|".join(str(c) for c in contagem.tolist())

        linhas.append(registro)
        if (i + 1) % 10 == 0:
            print(f"   {nome_grupo}: {i + 1}/{len(ds)}", flush=True)

    return linhas


def main(n_amostras=N_AMOSTRAS, n_imagens=N_IMAGENS, seed=SEED):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f">> device = {device}  |  N = {n_amostras} amostras por imagem", flush=True)
    print(f">> {n_imagens} imagens por grupo  |  pasta: {PASTA_TRABALHO}", flush=True)

    gerador = torch.Generator().manual_seed(seed)
    f_theta = ResNet50Imagenette().to(device).eval()
    for p in f_theta.parameters():
        p.requires_grad_(False)
    cabeca = carregar_cabeca(device)

    grupos = {
        "clean": DatasetAtacado("test", CSV_TEST_PRINCIPAL, "clean", n_max=n_imagens),
        "pgd": DatasetAtacado("test", CSV_TEST_PGD, "pgd", n_max=n_imagens),
    }

    linhas = []
    for nome, ds in grupos.items():
        print(f">> processando {nome} ({len(ds)} imagens x {n_amostras} amostras x 2 modos)...", flush=True)
        linhas.extend(processar_grupo(nome, ds, f_theta, cabeca, device, n_amostras, gerador))

    df = pd.DataFrame(linhas)
    saida = os.path.join(PASTA_TRABALHO, f"resultados_heatmap_n{n_amostras}.csv")
    df.to_csv(saida, index=False)
    print(f">> salvo em {saida}", flush=True)

    # ---------------------------------------------------------- resumo
    print(f"\n{'grupo':8s}  {'y_hat ok':>9s}  {'forca':>7s}  {'disp':>7s}"
          f"  {'ocl.conc':>9s}  {'ocl.voto':>9s}  {'rec.conc':>9s}  {'rec.voto':>9s}", flush=True)
    for nome in grupos:
        s = df[df["grupo"] == nome]
        print(f"{nome:8s}  {s['y_hat_correto'].mean():9.3f}  {s['forca_mapa'].mean():7.4f}"
              f"  {s['dispersao_mapa'].mean():7.4f}  {s['oclusao_concordancia'].mean():9.3f}"
              f"  {s['oclusao_voto_correto'].mean():9.3f}  {s['recorte_concordancia'].mean():9.3f}"
              f"  {s['recorte_voto_correto'].mean():9.3f}", flush=True)

    print("\n>> AMOSTRAS AUMENTADAS POR CLASSE (soma das N amostras de todas as imagens):", flush=True)
    for nome in grupos:
        s = df[df["grupo"] == nome]
        for modo in ["oclusao", "recorte"]:
            total = np.zeros(len(CLASSES_IMAGENETTE), dtype=int)
            for h in s[f"{modo}_histograma"]:
                total += np.array([int(v) for v in h.split("|")])
            print(f"   {nome}/{modo}: total={total.sum()}", flush=True)
            for c, qtd in enumerate(total):
                if qtd > 0:
                    print(f"      {CLASSES_IMAGENETTE[c]:20s}  {qtd:6d}  ({100 * qtd / total.sum():5.1f}%)", flush=True)

    print(f"\n>> rode a etapa 4 sobre {saida} para a análise de sinais.", flush=True)


if __name__ == "__main__":
    main()


>> device = cuda  |  N = 100 amostras por imagem
>> 40 imagens por grupo  |  pasta: /content/drive/MyDrive/adversarial_heatmap
>> processando clean (40 imagens x 100 amostras x 2 modos)...
   clean: 10/40
   clean: 20/40
   clean: 30/40
   clean: 40/40
>> processando pgd (40 imagens x 100 amostras x 2 modos)...
   pgd: 10/40
   pgd: 20/40
   pgd: 30/40
   pgd: 40/40
>> salvo em /content/drive/MyDrive/adversarial_heatmap/resultados_heatmap_n100.csv

grupo      y_hat ok    forca     disp   ocl.conc   ocl.voto   rec.conc   rec.voto
clean         0.975   0.2846   0.3666      0.966      1.000      0.876      0.950
pgd           0.000   0.3670   0.3553      0.930      0.075      0.270      0.600

>> AMOSTRAS AUMENTADAS POR CLASSE (soma das N amostras de todas as imagens):
   clean/oclusao: total=4000
      tench                    305  (  7.6%)
      English springer         288  (  7.2%)
      cassette player          101  (  2.5%)
      chain saw                400  ( 10.0%)
      church  

In [ ]:
# ETAPA 4
"""
ETAPA 4 -- descobre QUAL sinal separa melhor clean de pgd, e calibra o
limiar usando só o grupo clean.

Testa cada sinal candidato nas DUAS direções (alto=suspeito e
baixo=suspeito), sem presumir -- foi assim que descobrimos, em outros
experimentos, que a direção às vezes vem invertida da hipótese.


"""

import glob
import os
import pandas as pd
from sklearn.metrics import roc_auc_score

from comum import PASTA_TRABALHO

# ----------------------------------------------------- PARÂMETROS
CAMINHO_CSV = None    # None = pega o resultado mais recente automaticamente
PERCENTIL = 10.0      # percentil do clean para calibrar o limiar

SINAIS = [
    ("forca_mapa", "força média do mapa (sem augmentation)"),
    ("dispersao_mapa", "dispersão do mapa (sem augmentation)"),
    ("oclusao_concordancia", "concordância com y_hat sob oclusão"),
    ("recorte_concordancia", "concordância com y_hat sob recorte"),
    ("oclusao_n_classes_distintas", "nº de classes distintas sob oclusão"),
    ("recorte_n_classes_distintas", "nº de classes distintas sob recorte"),
]


def _achar_csv():
    if CAMINHO_CSV:
        return CAMINHO_CSV
    padrao = os.path.join(PASTA_TRABALHO, "resultados_heatmap_n*.csv")
    encontrados = sorted(glob.glob(padrao), key=os.path.getmtime)
    if not encontrados:
        raise FileNotFoundError(f"nenhum csv encontrado em {padrao} -- rode a etapa 3 antes.")
    return encontrados[-1]


def main(caminho_csv=None, percentil=PERCENTIL):
    caminho = caminho_csv or _achar_csv()
    print(f">> lendo {caminho}", flush=True)
    df = pd.read_csv(caminho)
    clean = df[df["grupo"] == "clean"]
    pgd = df[df["grupo"] == "pgd"]
    print(f">> {len(clean)} imagens clean, {len(pgd)} pgd", flush=True)

    y_true = [0] * len(clean) + [1] * len(pgd)
    resultados = []

    print(f"\n{'sinal':32s}  {'clean':>8s}  {'pgd':>8s}  {'AUROC':>7s}  direção", flush=True)
    for coluna, descricao in SINAIS:
        if coluna not in df.columns:
            continue
        valores = list(clean[coluna]) + list(pgd[coluna])
        auroc_alto = roc_auc_score(y_true, valores)
        auroc = max(auroc_alto, 1 - auroc_alto)
        direcao = "alto=suspeito" if auroc_alto >= 0.5 else "baixo=suspeito"
        resultados.append((auroc, coluna, direcao, descricao))
        print(f"{coluna:32s}  {clean[coluna].mean():8.4f}  {pgd[coluna].mean():8.4f}"
              f"  {auroc:7.3f}  {direcao}", flush=True)

    if not resultados:
        print(">> nenhum sinal encontrado no csv.", flush=True)
        return

    resultados.sort(reverse=True)
    melhor_auroc, melhor_coluna, melhor_direcao, melhor_desc = resultados[0]

    print(f"\n>> MELHOR SINAL: {melhor_coluna}  (AUROC={melhor_auroc:.3f}, {melhor_direcao})", flush=True)
    print(f"   {melhor_desc}", flush=True)

    # --------------------------------------- calibração só no clean
    if melhor_direcao == "baixo=suspeito":
        limiar = clean[melhor_coluna].quantile(percentil / 100)
        fpr = (clean[melhor_coluna] < limiar).mean()
        tpr = (pgd[melhor_coluna] < limiar).mean()
    else:
        limiar = clean[melhor_coluna].quantile(1 - percentil / 100)
        fpr = (clean[melhor_coluna] > limiar).mean()
        tpr = (pgd[melhor_coluna] > limiar).mean()

    print(f"\n>> limiar calibrado SÓ no clean (percentil {percentil:.0f}): {limiar:.4f}", flush=True)
    print(f"   falso positivo (clean marcado suspeito): {100 * fpr:.1f}%", flush=True)
    print(f"   detecção (pgd marcado suspeito):         {100 * tpr:.1f}%", flush=True)

    # ------------------------------------------------- robustez do voto
    print(f"\n>> ROBUSTEZ (voto majoritário corrige o rótulo?):", flush=True)
    print(f"{'grupo':8s}  {'y_hat':>7s}  {'oclusão':>8s}  {'recorte':>8s}", flush=True)
    for nome, sub in [("clean", clean), ("pgd", pgd)]:
        print(f"{nome:8s}  {sub['y_hat_correto'].mean():7.3f}"
              f"  {sub['oclusao_voto_correto'].mean():8.3f}"
              f"  {sub['recorte_voto_correto'].mean():8.3f}", flush=True)
    print("   (no pgd, voto > y_hat significa que a augmentation reverteu o ataque)", flush=True)


if __name__ == "__main__":
    main()

>> lendo /content/drive/MyDrive/adversarial_heatmap/resultados_heatmap_n100.csv
>> 40 imagens clean, 40 pgd

sinal                                clean       pgd    AUROC  direção
forca_mapa                          0.2846    0.3670    0.681  alto=suspeito
dispersao_mapa                      0.3666    0.3553    0.516  baixo=suspeito
oclusao_concordancia                0.9655    0.9295    0.595  baixo=suspeito
recorte_concordancia                0.8763    0.2700    0.949  baixo=suspeito
oclusao_n_classes_distintas         1.1750    1.4250    0.597  alto=suspeito
recorte_n_classes_distintas         2.2750    3.7500    0.758  alto=suspeito

>> MELHOR SINAL: recorte_concordancia  (AUROC=0.949, baixo=suspeito)
   concordância com y_hat sob recorte

>> limiar calibrado SÓ no clean (percentil 10): 0.5710
   falso positivo (clean marcado suspeito): 10.0%
   detecção (pgd marcado suspeito):         80.0%

>> ROBUSTEZ (voto majoritário corrige o rótulo?):
grupo       y_hat   oclusão   recorte
cl